## PFP Validation

written by Isobel Mawby (i.mawby1@lancaster.ac.uk)

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Imports
</div>

In [ ]:
import random
import uproot
import numpy as np
import math
import matplotlib.pyplot as plt
import awkward as ak

%matplotlib widget
from termcolor import colored, cprint

import Definitions
import ValidationFunc
import PFPValidationFunc

import os

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Config
</div>

In [ ]:
SHOW_PLOTS = False

<div class="alert alert-block alert-info" style="font-size: 18px;">
    File
</div>

In [ ]:
file_name = "/Users/isobel/Desktop/DUNE/2026/PandoraValidation/files/Validation_nu_1_EndDir.root"
plot_dir = '/Users/isobel/Desktop/DUNE/2026/PandoraValidation/PFPValPlots/'

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Lets open the file...
</div>

In [ ]:
file = uproot.open(file_name)

In [ ]:
event_tree = file['EventTree']
pfp_tree = file['PFPTree']
hierarchy_tree = file['HierarchyTree']

event_branches = event_tree.arrays(['Run', 'Subrun', 'Event', 'MCInt_IsCC', 'MCNu_PDG'], library="ak")

pfp_branches = pfp_tree.arrays(['Run', 'Subrun', 'Event',
                                'MCP_TruePDG', 'MCP_TrueEnergy', 'MCP_TrueVisEnergy', 'MCP_TrueThetaXZ', 'MCP_TrueThetaYZ',
                                'MCP_NMCHits2D', 'MCP_NMCHitsU', 'MCP_NMCHitsV', 'MCP_NMCHitsW',
                                'MCP_HasMatch', 'MCP_Length', 'MCP_Displacement',
                                'BM_IsTrack', 'BM_IsShower',
                                'BM_Completeness', 'BM_CompletenessU', 'BM_CompletenessV', 'BM_CompletenessW',
                                'BM_Purity', 'BM_PurityU', 'BM_PurityV', 'BM_PurityW',
                                'BM_VertexAcc', 'BM_Length', 'BM_Displacement', 
                                'ALT_Completeness', 'ALT_Purity', 'ALT_PDG', 'ALT_IsUpstreamHierarchy', 'ALT_IsSameMC'], library="ak")

hierarchy_branches = hierarchy_tree.arrays(['MC_HierarchyTier'], library="ak")

In [ ]:
print(pfp_tree.keys())

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Summary
</div>

In [ ]:
int_masks = Definitions.GetIntMasks(event_branches, pfp_branches)
pdg_masks = Definitions.GetPDGMasks(pfp_branches)
tier_masks = Definitions.GetTierMasks(hierarchy_branches)

# Apply custom def of reconstructable and reconstructed
# The tree constains some particles that we reconstructed, even if not initially deemed to be a target
pfp_target_mask = Definitions.GetIsTargetMask(pfp_branches)
pfp_reco_mask = Definitions.GetIsRecoMask(pfp_target_mask, pfp_branches)

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Add multiplicity
</div>

In [ ]:
multiplicity = ak.sum(pfp_target_mask, axis=1)
shower_multiplicity = ak.sum(pfp_target_mask & ((abs(pfp_branches['MCP_TruePDG']) == 11) | (pfp_branches['MCP_TruePDG'] == 22)), axis=1)
track_multiplicity = ak.sum(pfp_target_mask & (abs(pfp_branches['MCP_TruePDG']) != 11) & (pfp_branches['MCP_TruePDG'] != 22), axis=1)

# Match shape to PFP jagged arra
multiplicity = ak.broadcast_arrays(multiplicity, pfp_branches['MCP_TruePDG'])[0]
shower_multiplicity = ak.broadcast_arrays(shower_multiplicity, pfp_branches['MCP_TruePDG'])[0]
track_multiplicity = ak.broadcast_arrays(track_multiplicity, pfp_branches['MCP_TruePDG'])[0]

pfp_branches = ak.with_field(
    pfp_branches,
    multiplicity,
    "MCNu_Multiplicity"
)

pfp_branches = ak.with_field(
    pfp_branches,
    shower_multiplicity,
    "MCNu_ShowerMultiplicity"
)

pfp_branches = ak.with_field(
    pfp_branches,
    track_multiplicity,
    "MCNu_TrackMultiplicity"
)

multiplicity_mask = Definitions.GetMultiplicityMasks(pfp_branches) 

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Make directories
</div>

In [ ]:
#  MCP_var distributions
if not os.path.isdir(f'{plot_dir}/MCP/') :
    os.makedirs(f'{plot_dir}/MCP/')
for i_var in range(len(ValidationFunc.PFP_MCP_plotting_vars)) :
    if not os.path.isdir(f'{plot_dir}/MCP/{ValidationFunc.PFP_MCP_plotting_vars[i_var].tree_name}') :
        os.makedirs(f'{plot_dir}/MCP/{ValidationFunc.PFP_MCP_plotting_vars[i_var].tree_name}')

# BM_var distributions
if not os.path.isdir(f'{plot_dir}/BM/') :
    os.makedirs(f'{plot_dir}/BM/')
for i_var in range(len(ValidationFunc.PFP_BM_plotting_vars)) :
    if not os.path.isdir(f'{plot_dir}/BM/{ValidationFunc.PFP_BM_plotting_vars[i_var].tree_name}') :
        os.makedirs(f'{plot_dir}/BM/{ValidationFunc.PFP_BM_plotting_vars[i_var].tree_name}')

# ALT_var distributions
if not os.path.isdir(f'{plot_dir}/ALT/') :
    os.makedirs(f'{plot_dir}/ALT/')
for i_var in range(len(ValidationFunc.PFP_ALT_plotting_vars)) :
    if not os.path.isdir(f'{plot_dir}/ALT/{ValidationFunc.PFP_ALT_plotting_vars[i_var].tree_name}') :
        os.makedirs(f'{plot_dir}/ALT/{ValidationFunc.PFP_ALT_plotting_vars[i_var].tree_name}')
    if not os.path.isdir(f'{plot_dir}/ALT/{ValidationFunc.PFP_ALT_plotting_vars[i_var].tree_name}/Seg') :
        os.makedirs(f'{plot_dir}/ALT/{ValidationFunc.PFP_ALT_plotting_vars[i_var].tree_name}/Seg')

# Diff distributions
if not os.path.isdir(f'{plot_dir}/Diff/') :
    os.makedirs(f'{plot_dir}/Diff/')
for i_var in range(len(ValidationFunc.PFP_diff_plotting_vars)) :
    if not os.path.isdir(f'{plot_dir}/Diff/{ValidationFunc.PFP_diff_plotting_vars[i_var].true_tree_name}_{ValidationFunc.PFP_diff_plotting_vars[i_var].reco_tree_name}') :
        os.makedirs(f'{plot_dir}/Diff/{ValidationFunc.PFP_diff_plotting_vars[i_var].true_tree_name}_{ValidationFunc.PFP_diff_plotting_vars[i_var].reco_tree_name}')

# Track-shower
if not os.path.isdir(f'{plot_dir}/TrackShower/') :
    os.makedirs(f'{plot_dir}/TrackShower/')
for i_var in range(len(ValidationFunc.PFP_track_shower_plotting_vars)) :
    if not os.path.isdir(f'{plot_dir}/TrackShower/{ValidationFunc.PFP_track_shower_plotting_vars[i_var].tree_name}') :
        os.makedirs(f'{plot_dir}/TrackShower/{ValidationFunc.PFP_track_shower_plotting_vars[i_var].tree_name}')

# Efficiency
if not os.path.isdir(f'{plot_dir}/Efficiency/') :
    os.makedirs(f'{plot_dir}/Efficiency/')
for i_var in range(len(ValidationFunc.PFP_efficiency_vars)) :
    if not os.path.isdir(f'{plot_dir}/Efficiency/{ValidationFunc.PFP_efficiency_vars[i_var].tree_name}') :
        os.makedirs(f'{plot_dir}/Efficiency/{ValidationFunc.PFP_efficiency_vars[i_var].tree_name}')

<div class="alert alert-block alert-info" style="font-size: 18px;">
    Variables to plot
</div>

In [ ]:
PFPValidationFunc.CreateGraphs(plot_dir, pfp_target_mask, pfp_reco_mask, tier_masks, int_masks, pdg_masks, pfp_branches)